In [1]:
import pandas as pd

In [5]:
# Load the raw transaction dataset

file_path = "../data/raw/online_retail_II.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
# Check customer ID availability before cleaning

missing_customer_id = df["Customer ID"].isna().sum()
available_customer_id = df["Customer ID"].notna().sum()

print("Transactions with Customer ID:", available_customer_id)
print("Transactions without Customer ID:", missing_customer_id)
print(
    "Customer ID coverage:",
    round(available_customer_id / len(df) * 100, 2),
    "%"
)

Transactions with Customer ID: 824364
Transactions without Customer ID: 243007
Customer ID coverage: 77.23 %


In [7]:
# Keep transactions with a known customer ID

customer_df = df[df["Customer ID"].notna()].copy()

print("Original rows:", len(df))
print("Customer-level rows:", len(customer_df))
print("Rows removed:", len(df) - len(customer_df))

Original rows: 1067371
Customer-level rows: 824364
Rows removed: 243007


In [8]:
# Identify cancellation transactions in customer-level data

customer_cancellations = customer_df[
    customer_df["Invoice"].astype(str).str.startswith("C")
].copy()

print("Cancellation rows:", len(customer_cancellations))
print("Cancellation invoices:", customer_cancellations["Invoice"].nunique())

Cancellation rows: 18744
Cancellation invoices: 7901


In [9]:
# Check quantities in cancellation transactions

print(
    customer_cancellations["Quantity"].describe()
)

count    18744.000000
mean       -25.970604
std        821.040686
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64


In [10]:
# Remove cancellation transactions from customer-level data

purchase_df = customer_df[
    ~customer_df["Invoice"].astype(str).str.startswith("C")
].copy()

print("Customer-level rows:", len(customer_df))
print("Purchase rows:", len(purchase_df))
print("Cancellation rows removed:", len(customer_df) - len(purchase_df))

Customer-level rows: 824364
Purchase rows: 805620
Cancellation rows removed: 18744


In [11]:
# Check remaining negative quantities after removing cancellations

remaining_negative_quantity = (
    purchase_df["Quantity"] < 0
).sum()

print(
    "Remaining negative-quantity rows:",
    remaining_negative_quantity
)

Remaining negative-quantity rows: 0


In [12]:
# Check zero-price transactions after removing cancellations

zero_price_rows = (
    purchase_df["Price"] == 0
).sum()

print("Remaining zero-price rows:", zero_price_rows)

Remaining zero-price rows: 71


In [13]:
# Inspect remaining zero-price transactions

zero_price_purchases = purchase_df[
    purchase_df["Price"] == 0
].copy()

print(zero_price_purchases["Description"].value_counts(dropna=False).head(20))

Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
This is a test product.              2
REGENCY CAKESTAND 3 TIER             2
ROUND CAKE TIN VINTAGE GREEN         2
6 RIBBONS EMPIRE                     1
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
ANTIQUE GLASS HEART DECORATION       1
 FLAMINGO LIGHTS                     1
CHARLOTTE BAG , SUKI DESIGN          1
RETRO SPOT LARGE MILK JUG            1
VINTAGE GLASS COFFEE CADDY           1
CAST IRON HOOK GARDEN TROWEL         1
CAST IRON HOOK GARDEN FORK           1
AIRLINE BAG VINTAGE JET SET WHITE    1
HANGING METAL BIRD BATH              1
SET/5 RED SPOTTY LID GLASS BOWLS     1
DOORMAT HOME SWEET HOME BLUE         1
Name: count, dtype: int64


In [14]:
# Remove zero-price transactions

clean_df = purchase_df[
    purchase_df["Price"] > 0
].copy()

print("Before removing zero-price rows:", len(purchase_df))
print("After removing zero-price rows:", len(clean_df))
print("Zero-price rows removed:", len(purchase_df) - len(clean_df))

Before removing zero-price rows: 805620
After removing zero-price rows: 805549
Zero-price rows removed: 71


In [15]:
# Check duplicate transactions after initial cleaning

duplicate_rows = clean_df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 26124


In [16]:
# Inspect exact duplicate transactions

duplicates = clean_df[
    clean_df.duplicated(keep=False)
].sort_values(
    by=["Invoice", "StockCode", "InvoiceDate"]
)

print("Duplicate rows:", len(duplicates))
print("\nSample:")
print(duplicates.head(20))

Duplicate rows: 50836

Sample:
    Invoice StockCode                        Description  Quantity  \
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
363  489517     21912           VINTAGE SNAKES & LADDERS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
383  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED   